In [1]:
import os
import sys
import json
from tqdm.notebook import tqdm

In [2]:
mcGill_dir = '/storage/naumtsevalex/harmony/data/McGill-Billboard'

tree_lab_json_file = '/storage/naumtsevalex/harmony/data/results/00_test_folder_nSplits_10/woNochord_withNegSamples/woNochord_withNegSamples_labeling.json'
tree_lab_dir = os.path.dirname(tree_lab_json_file)
tree_name, tree_ext = os.path.splitext(os.path.basename(tree_lab_json_file))
tree_lab_mir_file = os.path.join(tree_lab_dir, f'{tree_name}.lab')
# gt_lab_mir_file = os.path.join(tree_lab_dir, f'gt_label.lab')


In [3]:
tree_name, tree_ext

('woNochord_withNegSamples_labeling', '.json')

In [4]:
import mir_eval
sys.path.append('./../../music-transformer')
from audio_chords.utils import calc_normalized_intesection_drop_nochord

_CHROMA_NOTES = ['C','Db','D','Eb','E','F','Gb','G','Ab','A','Bb','B']
_NO_CHORD = 'N'
_MAJMIN_CLASSES = [_NO_CHORD, *[f'{note}:maj' for note in _CHROMA_NOTES],
                   *[f'{note}:min' for note in _CHROMA_NOTES]]


def json_lab2mir_lab(out_labels, lab_fn):    
    if lab_fn: # dump labels to file
        str_labels = [f'{st}\t{ed}\t{_MAJMIN_CLASSES[chord_name]}'
                      for st, ed, chord_name in out_labels]
        with open(lab_fn, 'w') as f:
            for line in str_labels:
                f.write("%s\n" % line)

    return out_labels

def mir_lab2list_of_tuple(fn):
    timestamps, chord_labels = mir_eval.io.load_labeled_intervals(fn, comment='\n')
    # print(chord_labels)
    chord_labels_int = [_MAJMIN_CLASSES.index(chord_label_str) if chord_label_str in _MAJMIN_CLASSES else 0
                        for chord_label_str in chord_labels]
    res_st_en_ch = [(*timestamps[ind], chord_labels_int[ind]) for ind in range(len(timestamps))] 
    
    return res_st_en_ch
    

In [5]:
tree_json = json.load(open(tree_lab_json_file, 'r'))
for track_id, labels_dict in tqdm(tree_json.items()):
    gt_label = labels_dict['gt_labeling']
    pred_label = labels_dict['pred_labeling']
    score = labels_dict['score']
    
    mcGill_track_dir = os.path.join(mcGill_dir, f'{int(track_id):04d}')
    tree_lab_file = os.path.join(mcGill_track_dir, 'tree.lab')
    json_lab2mir_lab(pred_label, lab_fn=tree_lab_file)
    print(tree_lab_file)
    
    

  0%|          | 0/26 [00:00<?, ?it/s]

/storage/naumtsevalex/harmony/data/McGill-Billboard/0022/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0056/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0067/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0089/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0106/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0107/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0116/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0123/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0124/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0126/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0130/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0159/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0168/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0193/tree.lab
/storage/naumtsevalex/harmony/data/McGill-Billboard/0195/tree.lab
/storage/n

In [6]:
tree_json = json.load(open(tree_lab_json_file, 'r'))
tree_score = 0
autochord_score = 0

for track_id in tqdm(tree_json.keys()):
    mcGill_track_dir = os.path.join(mcGill_dir, f'{int(track_id):04d}')
    gt_lab_file = os.path.join(mcGill_track_dir, 'majmin.lab')
    tree_lab_file = os.path.join(mcGill_track_dir, 'tree.lab')
    autochord_lab_file = os.path.join(mcGill_track_dir, 'autochord.lab')
    print(tree_lab_file)
    
    gt_list = mir_lab2list_of_tuple(gt_lab_file)
    tree_list = mir_lab2list_of_tuple(tree_lab_file)
    autochord_list = mir_lab2list_of_tuple(autochord_lab_file)
    
    cur_tree_score = calc_normalized_intesection_drop_nochord(gt_list, tree_list)
    cur_autochord_score = calc_normalized_intesection_drop_nochord(gt_list, autochord_list)
    
    tree_score += cur_tree_score
    autochord_score += cur_autochord_score
    
    print(cur_tree_score, ' ', cur_autochord_score)
    
print()
print(tree_score / len(tree_json), autochord_score/ len(tree_json))

  0%|          | 0/26 [00:00<?, ?it/s]

/storage/naumtsevalex/harmony/data/McGill-Billboard/0022/tree.lab
0.834409489282226   0.8920634609977444
/storage/naumtsevalex/harmony/data/McGill-Billboard/0056/tree.lab
0.7520229733939632   0.8041865797195394
/storage/naumtsevalex/harmony/data/McGill-Billboard/0067/tree.lab
0.8026610127241329   0.8157129618166358
/storage/naumtsevalex/harmony/data/McGill-Billboard/0089/tree.lab
0.810925733523416   0.8865032980118043
/storage/naumtsevalex/harmony/data/McGill-Billboard/0106/tree.lab
0.7118491928924183   0.8105147088521143
/storage/naumtsevalex/harmony/data/McGill-Billboard/0107/tree.lab
0.6969635026400689   0.7569538171332025
/storage/naumtsevalex/harmony/data/McGill-Billboard/0116/tree.lab
0.821479510927957   0.8043235047062712
/storage/naumtsevalex/harmony/data/McGill-Billboard/0123/tree.lab
0.888947155498413   0.9320439070673207
/storage/naumtsevalex/harmony/data/McGill-Billboard/0124/tree.lab
0.849674008695193   0.8959443602148356
/storage/naumtsevalex/harmony/data/McGill-Billboard